# W07-H2 Exercise: Attention arithmetic, masks, and interpretation

Machine Learning in Engineering Science | NCKU / LAiMM | Fall 2026

**Date:** 2026-10-19. **Exercise time:** 20 minutes within the class/practice allocation, not extra time after class.

**Classroom formative practice. No new weekly graded submission is created.**

## How to work independently
1. In Colab, save a copy to your Drive and use a CPU runtime. In Jupyter, open this notebook and select a Python kernel.
2. Run the complete worked walkthrough below from top to bottom. Read every Markdown checkpoint; the walkthrough is included so no previous notebook state is needed.
3. At the Independent Exercise section, write each prediction in the response cell before running its experiment.
4. Run the starter experiment, make the requested controlled change, and record the actual result. Edit only the declared factor; preserve split rules and preprocessing.
5. Explain one mechanism and one limitation. Export the evidence JSON and discuss it with a partner.

The core exercises need no paid API, private dataset, or GPU. The agent units use clearly labeled offline simulators, not real LLM calls.

**Reproducibility boundary:** the original guided notebooks were previously executed; the new extension cells have not yet been execution-tested in this revision. Report unexpected behavior rather than claiming a check passed.


# W07 H2: Why Attention, and How It Works

Machine Learning in Engineering Science | NCKU + LAiMM | Fall 2026

**Teaching date:** Monday 19 October 2026. **Guided notebook time:** 25 minutes within a 60-minute module.

### What you will be able to explain
- Explain the fixed-context bottleneck and query-dependent retrieval.
- Compute scores, softmax weights, and a weighted sum by hand.
- Distinguish masks, attention weights, and causal explanations.

### Start here, even if this is your first notebook
1. In Colab choose **Runtime > Change runtime type > CPU**. In Jupyter choose a Python 3 kernel.
2. Run cells from top to bottom with **Shift+Enter**. Read the Markdown before each code cell.
3. The setup installs only missing free packages. A network connection is needed for first installation; the data itself is generated locally.
4. If a variable is undefined, restart the kernel and run all earlier cells. Do not skip a setup cell.
5. Save a personal copy before changing parameters. Every core cell is complete: there are no hidden files, tokens, paid services, or unfinished code blocks.

**Data:** synthetic teaching data, not engineering measurements. Record your actual output; a model is not required to outperform a baseline.
**Practice policy:** in-class exercise and optional Homework extension, not a new weekly graded assignment.


## Mechanism reference: compare, normalize, retrieve

For $Q$ of shape $(N_q,d_k)$, $K$ of shape $(N_k,d_k)$, and $V$ of shape $(N_k,d_v)$:

$$S=QK^\top/\sqrt{d_k},\quad A=\operatorname{softmax}_{\mathrm{keys}}(S+M),\quad C=AV.$$

Mask $M$ is zero for allowed entries and negative infinity for blocked entries. Rows of $A$ are receiving queries and columns are source keys. Every valid row sums to one. $C$ has shape $(N_q,d_v)$. Keep key rows paired with their value rows.

Attention addresses a final-state encoder's fixed-context bottleneck by retrieving earlier states for each output query. It can be attached to an LSTM or GRU. Self-attention derives Q,K,V from one sequence; cross-attention derives queries from one stream and keys/values from another.



## 1. Set up a numerical experiment

Attention can be understood without training a large model. First we use NumPy so each intermediate value is visible.


In [ ]:
import sys, subprocess, importlib.util, platform, json, random, hashlib
from pathlib import Path
required = {"numpy": "numpy", "matplotlib": "matplotlib", "sklearn": "scikit-learn"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
import numpy as np
import matplotlib.pyplot as plt
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)
print("Python:", platform.python_version(), "NumPy:", np.__version__, "Seed:", SEED)


**Checkpoint.** Run this notebook independently of H1. No saved H1 model is required.


## 2. Start with a query and an addressable memory

Q asks what to retrieve, K describes how each memory item matches the query, and V contains the information to combine. These numbers are hand-designed teaching vectors, not learned semantic embeddings. Q has shape (1,2), K is (3,2), and V is (3,2).


In [ ]:
Q = np.array([[1.,0.]])
K = np.array([[1.,0.],[0.,1.],[1.,1.]])
V = np.array([[10.,0.],[0.,10.],[10.,10.]])
labels = ["memory A","memory B","memory C"]
print("Q:",Q,"\nK:\n",K,"\nV:\n",V)
plt.scatter(K[:,0],K[:,1],s=120)
for name, point in zip(labels,K): plt.annotate(name,point,xytext=(5,7),textcoords="offset points")
plt.arrow(0,0,Q[0,0],Q[0,1],width=.01,color="tab:red",length_includes_head=True)
plt.xlim(-.2,1.4); plt.ylim(-.2,1.4); plt.xlabel("Key dimension 1"); plt.ylabel("Key dimension 2"); plt.show()


**Checkpoint.** A and C have the same dot product with this query. A key is an address; its associated value need not equal the key.


## 3. Calculate scaled scores

For each query, compute its dot product with every key, then divide by sqrt(d_k). This controls score scale as the key dimension grows under a unit-variance coordinate assumption. It does not normalize vector lengths.


In [ ]:
scores = Q @ K.T / np.sqrt(K.shape[-1])
print("Raw dot products:",Q@K.T)
print("Scaled scores:",scores)
assert np.allclose(scores,[[1/np.sqrt(2),0,1/np.sqrt(2)]])


**Checkpoint.** The scaled score row is approximately [0.7071, 0, 0.7071]. Scores are not yet probabilities and may be negative.


## 4. Convert scores into a row of weights

Softmax subtracts the row maximum for numerical stability, exponentiates, and divides by the row sum. Subtracting a shared constant changes no probability. Each query row must sum to 1 over the available keys.


In [ ]:
def softmax(z):
    shifted = z-z.max(axis=-1,keepdims=True)
    exp = np.exp(shifted)
    return exp/exp.sum(axis=-1,keepdims=True)
weights = softmax(scores)
print("Weights:",weights,"Row sum:",weights.sum(axis=-1))
assert np.allclose(weights.sum(axis=-1),1)
plt.bar(labels,weights[0],color=["#1e8798","#7f2635","#f3b23a"])
plt.ylim(0,1); plt.ylabel("Attention weight"); plt.show()


**Checkpoint.** The approximate weights are [0.4011, 0.1978, 0.4011]. B still has positive weight even though its raw score is zero.


## 5. Retrieve a weighted combination of values

The output is weights @ V. Each output dimension combines the same weights with the corresponding value coordinates. There is no argmax selection: multiple values contribute.


In [ ]:
context = weights @ V
print("Weighted contributions:\n",weights[0,:,None]*V)
print("Context vector:",context)
assert context.shape == (1,2)
assert np.allclose(context,(weights[0,:,None]*V).sum(axis=0,keepdims=True))


**Checkpoint.** The context is approximately [8.0222, 5.9889]. Explain which memory items contribute to each coordinate before moving on.


## 6. Change only the query

The memory stays fixed but the task changes. Try Q=[0,1] and Q=[-1,0]. Compute your predicted ranking before running. Temperature below 1 sharpens and above 1 softens the distribution; standard scaled dot-product attention uses temperature=1.


In [ ]:
queries = np.array([[1.,0.],[0.,1.],[-1.,0.]])
temperature = 1.0
assert temperature > 0
W = softmax((queries@K.T/np.sqrt(2))/temperature)
print("Contexts:\n",W@V)
fig,ax=plt.subplots(figsize=(6,3))
im=ax.imshow(W,cmap="YlGnBu",vmin=0,vmax=1)
ax.set_xticks(range(3),labels); ax.set_yticks(range(3),["q=[1,0]","q=[0,1]","q=[-1,0]"])
for row in range(3):
    for col in range(3): ax.text(col,row,f"{W[row,col]:.2f}",ha="center",va="center",color="black")
fig.colorbar(im,ax=ax); plt.show()


**Checkpoint.** A query with negative first coordinate reduces the weights of keys with positive first coordinate. Attention retrieval depends on the query, not on one universal ranking.


## 7. Add causal masking before softmax

For a token at position t, a causal decoder may access positions <=t, including itself. Set future scores to negative infinity before softmax; then their exponentials are zero. Every row must retain at least one allowed key.


In [ ]:
X_demo = np.array([[1.,0.],[0.,1.],[1.,1.]])
self_scores = X_demo@X_demo.T/np.sqrt(2)
allowed = np.tril(np.ones((3,3),dtype=bool))
masked_scores = np.where(allowed,self_scores,-np.inf)
causal_weights = softmax(masked_scores)
print("Allowed (query rows, key columns):\n",allowed.astype(int))
print("Causal attention:\n",causal_weights)
assert np.all(causal_weights[~allowed] == 0)
assert np.allclose(causal_weights.sum(axis=-1),1)
plt.imshow(causal_weights,cmap="YlGnBu",vmin=0,vmax=1)
plt.xlabel("Key/source position"); plt.ylabel("Query/destination position"); plt.colorbar(); plt.show()


**Checkpoint.** The first row must be [1,0,0]. Do not apply softmax before masking, and do not confuse the direction of rows and columns.


## 8. Separate padding from future masking

A padding mask hides nonexistent tokens, while a causal mask hides future tokens. Combine the two permissions. For padded query rows, the output is normally excluded from the loss rather than interpreted as a real prediction.


In [ ]:
is_real_key = np.array([True,True,False])
combined_allowed = allowed & is_real_key[None,:]
assert combined_allowed.any(axis=-1).all()
combined_weights = softmax(np.where(combined_allowed,self_scores,-np.inf))
print(combined_weights)
assert np.all(combined_weights[:,2] == 0)


**Checkpoint.** No query can retrieve the padded final key. An all-masked row would make softmax undefined; prevent that case rather than silently treating NaN as an answer.


## 9. Verify the same calculation with PyTorch

The high-level SDPA API uses True for allowed entries in a Boolean mask. Other PyTorch APIs use different mask conventions; always read the called API. We use an explicit implementation for comparison and disable dropout.


In [ ]:
if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch"])
import torch
from torch import nn
import copy, time
torch.set_num_threads(1)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
device = torch.device("cpu")
print("PyTorch:", torch.__version__, "Device:", device)

import torch.nn.functional as F
q=torch.tensor(Q,dtype=torch.float32)[None,None,:,:]
k=torch.tensor(K,dtype=torch.float32)[None,None,:,:]
v=torch.tensor(V,dtype=torch.float32)[None,None,:,:]
torch_context = F.scaled_dot_product_attention(q,k,v,dropout_p=0.0)
assert np.allclose(torch_context.numpy()[0,0],context,atol=1e-5)
print("NumPy and PyTorch agree:",torch_context[0,0])


**Checkpoint.** The parity assertion must pass. This is an implementation check for the chosen example, not evidence that attention has learned language.


## 10. Test what an attention plot can explain

Weights describe a computation, but causal importance requires an intervention. Hold Q and K fixed and change one V; weights remain unchanged while output changes. Then remove a memory item by masking it and observe that normalization also changes.


In [ ]:
V_changed=V.copy(); V_changed[1]=[100.,100.]
changed_context=weights@V_changed
removed_weights=softmax(np.where(np.array([[True,False,True]]),scores,-np.inf))
print("Original weights:",weights)
print("Same weights, changed values:",changed_context)
print("Remove B and renormalize:",removed_weights@V)
assert not np.allclose(changed_context,context)
evidence={"Q":Q.tolist(),"K":K.tolist(),"V":V.tolist(),"scores":scores.tolist(),
          "weights":weights.tolist(),"context":context.tolist(),
          "causal_weights":causal_weights.tolist(),"seed":SEED,
          "interpretation":"Attention weights are routing weights, not proof of causal importance."}
(ARTIFACTS/"w07-h2-attention.json").write_text(json.dumps(evidence,indent=2))


**Checkpoint.** State exactly what the heatmap supports: where the chosen computation routes information. Do not claim it proves why a real-world outcome occurred.


## 11. Independent practice and solutions
1. Set temperature to 0.5 and 2.0 in Step 6. Predict and then compare the sharpness, keeping Q, K, V fixed.
2. Replace all keys by the same vector. **Expected:** every available key gets equal weight, regardless of query; masked keys still get zero.
3. Make one value very large. **Expected:** the context may change greatly with no change to attention weights.
4. Explain why additive attention can be attached to an LSTM encoder-decoder. **Answer:** a decoder query can score all encoder states; attention does not require removing recurrence.
5. Explain self-attention versus cross-attention. **Answer:** self-attention derives Q,K,V from one sequence; cross-attention derives Q from one stream and K,V from another.

**Stop when:** your evidence JSON contains scores, weights, context, and a causal-mask assertion; explain all matrix axes to a partner.

**Bridge to H3:** a Transformer combines attention with positional information, a tokenwise feed-forward network, residual paths, and normalization. Attention alone is not the complete architecture.

### Sources
- [Bahdanau et al. (2014/2015), joint alignment and translation](https://arxiv.org/abs/1409.0473)
- [Vaswani et al. (2017), Attention Is All You Need](https://arxiv.org/abs/1706.03762)
- [PyTorch scaled dot-product attention API](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)
- [3Blue1Brown attention lesson](https://www.3blue1brown.com/lessons/attention/)


## Independent Exercise: W07-H2

Complete the three tasks below after the walkthrough. The starter cells produce observations, not your written interpretation. Fill the response fields in the next cell. Save separate records for the original run and any controlled change.

In [ ]:
exercise_id="W07-H2"
observations={}
responses={
    'task1_prediction':'', 'task1_explanation':'',
    'task2_prediction':'', 'task2_explanation':'',
    'task3_prediction':'', 'task3_explanation':'',
    'controlled_change':'', 'limitation':'', 'ai_use_and_checks':''
}
# Fill these strings before and after each experiment. Use 'none' if no AI assistance was used.
print('Exercise record initialized:', exercise_id)

### Task 1: Calculate retrieval coordinate by coordinate

Predict the three weights and the two output coordinates. Use the contributions table to explain which value supplies each coordinate.

**Steps**
1. Write your prediction in `responses['task1_prediction']` before running.
2. Run the starter cell; read every printed quantity and retain the observation.
3. Make the controlled comparison requested above. Record its configuration in `responses['controlled_change']`.
4. Write a mechanism-based explanation in `responses['task1_explanation']`; do not merely repeat a score.

**Checkpoint:** Weights are about [0.4011,0.1978,0.4011]; the context is about [8.0222,5.9889]. The sum must be performed separately for each coordinate.

In [ ]:
q_ex=np.array([[1.,0.]])
k_ex=np.array([[1.,0.],[0.,1.],[1.,1.]])
v_ex=np.array([[10.,0.],[0.,10.],[10.,10.]])
s_ex=q_ex@k_ex.T/np.sqrt(2)
w_ex=np.exp(s_ex-s_ex.max()); w_ex/=w_ex.sum(axis=-1,keepdims=True)
contributions=w_ex[0,:,None]*v_ex
observations['task1']={'weights':w_ex.tolist(),'contributions':contributions.tolist(),'context':(w_ex@v_ex).tolist()}
print(observations['task1'])

### Task 2: Mask before normalizing

Combine a causal mask with a padded last key. Check permitted-row sums and blocked weights. Do not mask a previously normalized matrix without renormalizing.

**Steps**
1. Write your prediction in `responses['task2_prediction']` before running.
2. Run the starter cell; read every printed quantity and retain the observation.
3. Make the controlled comparison requested above. Record its configuration in `responses['controlled_change']`.
4. Write a mechanism-based explanation in `responses['task2_explanation']`; do not merely repeat a score.

**Checkpoint:** Every row sums to one and the padded final key has zero weight. Each row must retain at least one permitted key.

In [ ]:
raw_scores=np.array([[1.,0.,1.],[0.,1.,1.],[1.,1.,2.]])/np.sqrt(2)
allowed_ex=np.tril(np.ones((3,3),dtype=bool)) & np.array([True,True,False])[None,:]
masked_ex=np.where(allowed_ex,raw_scores,-np.inf)
a_ex=np.exp(masked_ex-masked_ex.max(axis=-1,keepdims=True)); a_ex/=a_ex.sum(axis=-1,keepdims=True)
observations['task2']={'weights':a_ex.tolist(),'row_sums':a_ex.sum(axis=-1).tolist(),'blocked_weights_zero':bool(np.all(a_ex[~allowed_ex]==0))}
print(observations['task2'])

### Task 3: Separate routing from causal explanation

Change only the second value vector. Predict whether weights change and whether the retrieved context changes. Explain the scope of this intervention.

**Steps**
1. Write your prediction in `responses['task3_prediction']` before running.
2. Run the starter cell; read every printed quantity and retain the observation.
3. Make the controlled comparison requested above. Record its configuration in `responses['controlled_change']`.
4. Write a mechanism-based explanation in `responses['task3_explanation']`; do not merely repeat a score.

**Checkpoint:** Q and K are fixed, so these weights stay fixed while context changes. This tests a computation; it does not establish causation in a physical system.

In [ ]:
changed_v=v_ex.copy(); changed_v[1]=[100.,100.]
observations['task3']={'original_context':(w_ex@v_ex).tolist(),'changed_context':(w_ex@changed_v).tolist(),'same_weights':True}
print(observations['task3'])

### Finish: preserve and explain

Fill all response strings, then run the export cell. A draft file is intentionally labeled incomplete; a complete response record is not automatically a scientifically correct result. Download the JSON before the Colab session ends. Share it with a partner for explain-back, not as an additional weekly submission.

**Exit questions:** What changed? Which observation supports your explanation? What does this experiment not establish?

In [ ]:
missing_responses=[key for key,value in responses.items() if not str(value).strip()]
exercise_record={'exercise':exercise_id,'status':'draft' if missing_responses else 'ready for peer discussion','responses':responses,'observations':observations,'missing_responses':missing_responses,'environment':{'python':platform.python_version(),'numpy':np.__version__},'data_scope':'course-provided or synthetic teaching data'}
exercise_folder=Path('artifacts')/'exercises'; exercise_folder.mkdir(parents=True,exist_ok=True)
exercise_path=exercise_folder/(exercise_id+'-evidence.json')
exercise_path.write_text(json.dumps(exercise_record,indent=2,default=lambda value:value.tolist() if hasattr(value,'tolist') else str(value)))
print('Saved:',exercise_path)
print('Status:',exercise_record['status'])
print('Fields still to complete:',missing_responses)